# Lab 2. Titanic survival classification

Predict whether a passenger survived (`Survived = 1`) from the passenger list.

- **Data:** [Titanic Dataset](https://www.kaggle.com/datasets/yasserh/titanic-dataset), 891 passengers.
- **Metric:** F1 for the class `Survived = 1`, required level **> 0.70**; accuracy, precision and recall as well.
- **Plan:** data → split → EDA → preprocessing → baseline and five models → comparison → feature importance → error analysis.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 72

SEED = 42
TARGET = "Survived"

## 1. Data

In [ ]:
titanic = pd.read_csv("data/titanic.csv")
same_profile = titanic.drop(columns=["PassengerId", "Name"]).duplicated().sum()
print(f"passengers: {len(titanic)}, rows identical apart from id and name: {same_profile}")
print(f"survived: {titanic[TARGET].mean():.1%}")
pd.DataFrame(
    {
        "dtype": titanic.dtypes.astype(str),
        "missing": titanic.isna().sum(),
        "unique": titanic.nunique(),
    }
).T

**Findings**

- 891 passengers. Every row is a different person; 15 rows repeat another one in everything except id and name (e.g. relatives on one ticket), which is normal. 38% survived: the classes are imbalanced, so accuracy alone is misleading — "everybody died" already gives 62%.
- **Missing values:** `Age` (177, 20%) — imputed with the median and flagged by an indicator; `Embarked` (2) — imputed with the most frequent port; `Cabin` (687, 77%) — dropped. Too little is known about it, and the fact that it is known mostly repeats the class: cabins were recorded for first-class passengers.
- **Columns that identify a passenger** and would only let the model memorize objects:
  - `PassengerId` is a row number;
  - `Name` is unique for every passenger;
  - `Ticket` has 681 values for 891 rows, and a shared ticket marks a family or travel group.
  
  They are not used as features. The only thing taken from `Name` is the **title** (Mr, Mrs, Miss, Master), which is a category rather than an identity.
- **Need processing:** `Sex`, `Embarked` and the title are categorical (one-hot encoding). `SibSp` and `Parch` are combined into `FamilySize`.

In [ ]:
titanic["Title"] = (
    titanic["Name"]
    .str.extract(r",\s*([^.]+)\.", expand=False)
    .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
)
titanic.loc[~titanic["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Title"] = "Rare"
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1

NUMERIC = ["Pclass", "Age", "Fare", "FamilySize"]
CATEGORICAL = ["Sex", "Embarked", "Title"]
FEATURES = NUMERIC + CATEGORICAL
titanic["Title"].value_counts().to_frame("count").T

## 2. Train/test split

- **80/20** (712 / 179) with `stratify=Survived`, so both parts keep 38% survivors. `random_state=42`.
- All models use this one split. Imputers, encoders and scalers live inside a `Pipeline`, so they are fitted on the training data only — inside every CV fold as well.
- The test set is used once, for the final table. The best model is chosen by 5-fold stratified CV on train.

In [ ]:
train, test = train_test_split(titanic, test_size=0.2, stratify=titanic[TARGET], random_state=SEED)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"train: {len(train)}, survived {y_train.mean():.1%}")
print(f"test:  {len(test)}, survived {y_test.mean():.1%}")

## 3. Exploratory analysis

On the training part only.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, column in zip(
    axes.flat, ["Sex", "Pclass", "Title", "FamilySize", "Embarked"], strict=False
):
    sns.barplot(data=train, x=column, y=TARGET, errorbar=None, color="#4393c3", ax=ax)
    ax.set_ylabel("survival rate")
sns.histplot(
    data=train,
    x="Age",
    hue=TARGET,
    bins=30,
    multiple="stack",
    palette={0: "#d6604d", 1: "#4393c3"},
    ax=axes[1, 2],
)
fig.tight_layout()
plt.show()

train.pivot_table(index="Sex", columns="Pclass", values=TARGET, aggfunc="mean")

**Findings**

- **Sex is the strongest factor:** 74% of women survived vs 19% of men — "women and children first".
- **Class:** 65% / 45% / 24% for 1st / 2nd / 3rd class; first-class cabins were closer to the boats. Sex and class interact: almost all 1st and 2nd class women survived (97%, 91%), while 3rd class women survived only 50% of the time.
- **Title** combines sex and age: `Master` (boys, median age 3) survived 55% of the time, adult men (`Mr`) only 15%. The title therefore adds information about children that sex alone does not have.
- **Age:** children aged 12 or younger survived 60% of the time (55 children), passengers over 60 only 19% (but there are just 16 of them); in between age matters little.
- **Family:** passengers travelling alone (30%) and large families of 5+ (16%) survived less often than small families of 2–4 (59%).
- **Embarked:** Cherbourg 56% vs Southampton 33%. This is mostly the class: many 1st class passengers boarded at Cherbourg, so the port is not a reason by itself.
- `Fare` is another form of class (median 58 for 1st class vs 8 for 3rd).

## 4. Preprocessing and models

| Step | Numeric: `Pclass`, `Age`, `Fare`, `FamilySize` | Categorical: `Sex`, `Embarked`, `Title` |
| --- | --- | --- |
| Missing values | median + "age was missing" indicator (the median is robust to outliers; the indicator keeps the fact that age is unknown, which matters: 29% survival vs 38% overall) | most frequent value |
| Encoding | — | one-hot |
| Scaling | standardization, only for SVM and KNN | not needed: one-hot columns are already 0/1 |

`Pclass` is kept as a number because it is **ordinal** (1st > 2nd > 3rd class). Trees split it with a threshold anyway, and for SVM/KNN the distance between classes 1 and 3 being larger than between 1 and 2 is meaningful.

**Where scaling matters:**

- **SVM and KNN** work with distances between objects. Unscaled, `Fare` (0–512) would dominate `Pclass` (1–3) and the 0/1 columns, so the model would effectively look only at the fare. After standardization the numeric features have a spread of about one unit, comparable with the 0/1 columns.
- **Trees** compare one feature with a threshold at a time, so scale does not affect them.

**Models** use default hyperparameters; tuning is the topic of lab 3.

- **Decision Tree:** a sequence of if/else splits.
- **Random Forest:** 500 trees on random subsamples, majority vote.
- **Gradient Boosting:** shallow trees, each correcting the errors of the previous ones.
- **SVM:** the widest boundary between the classes, with an RBF kernel for non-linear boundaries.
- **KNN:** the majority class among the 5 nearest passengers.
- **Baseline:** `DummyClassifier`, always the most frequent class (died).

All models predict "survived" when their score passes the default threshold (probability 0.5, or the SVM boundary).

In [ ]:
def preprocessor(scale: bool, numeric: list[str] = NUMERIC) -> ColumnTransformer:
    numeric_steps = [SimpleImputer(strategy="median", add_indicator=True)]
    if scale:
        numeric_steps.append(StandardScaler())
    return ColumnTransformer(
        [
            ("num", make_pipeline(*numeric_steps), numeric),
            (
                "cat",
                make_pipeline(
                    SimpleImputer(strategy="most_frequent"),
                    OneHotEncoder(handle_unknown="ignore"),
                ),
                CATEGORICAL,
            ),
        ],
        verbose_feature_names_out=False,
    )


models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Decision Tree": make_pipeline(preprocessor(False), DecisionTreeClassifier(random_state=SEED)),
    "Random Forest": make_pipeline(
        preprocessor(False), RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
    ),
    "Gradient Boosting": make_pipeline(
        preprocessor(False), GradientBoostingClassifier(random_state=SEED)
    ),
    "SVM": make_pipeline(preprocessor(True), SVC(random_state=SEED)),
    "KNN": make_pipeline(preprocessor(True), KNeighborsClassifier()),
}

## 5. Metrics and comparison

- **Accuracy** — the share of correct answers.
- **Precision** — the share of passengers predicted as survivors who really survived.
- **Recall** — the share of real survivors that the model found.
- **F1** — the harmonic mean of precision and recall. It is high only when both are high, so the trivial "everybody died" gets 0.

The table shows train F1, CV F1 (5 folds on train) and the test metrics.

In [ ]:
def scores(y_true: pd.Series, y_pred: pd.Series) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }


rows, predictions = {}, {}
for name, model in models.items():
    cv_f1 = cross_val_score(model, X_train, y_train, cv=CV, scoring="f1").mean()
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    rows[name] = {
        "train F1": f1_score(y_train, model.predict(X_train)),
        "CV F1": cv_f1,
        **scores(y_test, predictions[name]),
    }
results = pd.DataFrame(rows).T
results["F1 > 0.70"] = results["F1"] > 0.70
results

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (name, y_pred) in zip(axes.flat, predictions.items(), strict=True):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=["died", "survived"], colorbar=False, cmap="Blues", ax=ax
    )
    ax.set_title(name)
    ax.grid(False)
fig.tight_layout()
plt.show()

**Findings: metrics**

- **The baseline has 62% accuracy but F1 = 0:** it never predicts a survivor, which is why accuracy is not the main metric.
- **All five models pass F1 > 0.70.** By CV the best are SVM (0.749) and Gradient Boosting (0.747); on the test SVM is also the best (F1 0.752, accuracy 0.827).
- **Decision Tree overfits:** train F1 0.98, CV 0.73, test 0.745. An unrestricted tree grows until it memorizes almost every training passenger, and on new passengers it loses about a quarter of that. Its test F1 happens to be close to the others, but its CV score is the lowest among the trees.
- **Random Forest** has the same train F1 (0.98, its trees are also deep) and almost the same CV (0.732). On this small dataset voting does not add much, because the signal is a few simple rules.
- **Gradient Boosting vs Random Forest:** boosting is better on CV (0.747 vs 0.732) and overfits less (train 0.88 vs 0.98), because its trees are shallow and added gradually. On the test the order flips (0.738 vs 0.741), but the difference is 0.003 — less than one passenger out of 69 survivors — so it is noise. CV over 5 folds is the more reliable comparison.
- **SVM** builds the smoothest boundary and overfits least (train 0.78 vs CV 0.75). It is the best on CV; on the test the four best models are within noise of each other (≤ 0.014 in F1).
- **KNN is the weakest** (CV 0.718, recall 0.65): a vote of only 5 neighbors is noisy, and in the regions where survivors and non-survivors mix (3rd class women, men in 1st class) the majority often says "died".

**Findings: confusion matrices** (test: 110 died, 69 survived)

| Model | FP (died, predicted survived) | FN (survived, predicted died) | Character |
| --- | --- | --- | --- |
| Decision Tree | 17 | 18 | balanced errors, the most FP |
| Random Forest | 16 | 19 | similar to the tree |
| Gradient Boosting | 13 | 21 | more cautious |
| SVM | 9 | 22 | most precise: rarely invents a survivor |
| KNN | 11 | 24 | misses the most survivors |

- Every model makes at least as many FN as FP, so recall is below precision: the models predict survival only for clear cases (women, 1st/2nd class, children), and survivors among men are missed.
- **FP vs FN in this task:**
  - FP — the model says "survived" for someone who died;
  - FN — it misses a survivor.
  
  If the model were used, for example, to plan a search for survivors, FN would be worse: a person who may be alive would not be looked for. A lower decision threshold would then trade FP for fewer FN. For an overall estimate both errors matter equally, which is what F1 balances.

## 6. Feature importance

In [ ]:
importance = pd.DataFrame(
    {
        name: pd.Series(
            models[name][-1].feature_importances_,
            index=models[name][0].get_feature_names_out(),
        )
        for name in ["Decision Tree", "Random Forest", "Gradient Boosting"]
    }
).sort_values("Random Forest", ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (name, color) in zip(
    axes, zip(importance.columns, ["#f4a582", "#5aae61", "#8073ac"], strict=True), strict=True
):
    importance[name].iloc[::-1].plot.barh(ax=ax, color=color)
    ax.set_title(name)
fig.tight_layout()
plt.show()


def source_feature(column: str) -> str:
    return "Age" if column.startswith("missingindicator") else column.split("_")[0]


importance.groupby(source_feature).sum().sort_values("Random Forest", ascending=False)

**Findings**

1. **Grouped by original feature** (one-hot columns summed, last table), the most important factors in all three models are **sex + title** (DT 0.37, RF 0.34, GB 0.51), then **fare**, **age** and **class**. Family size and port are minor.
2. **DT vs RF vs GB:**
   - The **Decision Tree** puts a third of its importance on `Title_Mr` (0.32). Its first split is "adult man or not", which repeats `Sex`, so `Sex_*` gets almost nothing — the information was already used.
   - **Gradient Boosting** relies on `Title_Mr` even more (0.42), then on `Fare` (0.19) and `Pclass` (0.12).
   - **Random Forest** spreads the importance between `Title_Mr`, `Sex_male` and `Sex_female`, which carry the same signal. Each tree sees a random subset of features, so sometimes one is used and sometimes another.
3. **Column by column, `Fare` and `Age` come first in RF and right after `Title_Mr` in DT.** This is partly an artifact: continuous features offer many split points, and deep trees use them to separate individual passengers. Impurity-based importance overrates such features. Boosting, with shallow trees, ranks them lower.
4. **Against EDA:** the order matches — sex/title first, then class (and fare as its proxy) and age. `Embarked` is almost unused; its effect in EDA was explained by class.

**Is an important feature the cause of survival? No.** Importance shows what the model uses to predict, not what causes survival.

- `Fare` did not save anyone. It is a marker of class, and class means cabin location and access to the boats.
- `Embarked` correlates with survival only through class.
- Even for sex the cause is not biology but the evacuation rule "women and children first".

The data is observational, and correlated features (fare and class, title and sex) share or take over each other's importance.

## 7. Error analysis

In [ ]:
best_name = results.drop(index="Baseline")["CV F1"].idxmax()
best = models[best_name]
errors = test.assign(
    predicted=predictions[best_name],
    score=best.decision_function(X_test)
    if hasattr(best, "decision_function")
    else best.predict_proba(X_test)[:, 1] - 0.5,
)
errors["error"] = "correct"
errors.loc[(errors[TARGET] == 0) & (errors["predicted"] == 1), "error"] = "FP"
errors.loc[(errors[TARGET] == 1) & (errors["predicted"] == 0), "error"] = "FN"
print(f"best model by CV F1: {best_name}")

display(pd.crosstab([errors["Sex"], errors["Pclass"]], errors["error"], margins=True))
pd.crosstab(errors["Title"], errors["error"])

In [ ]:
errors[errors["error"] != "correct"].sort_values("score")[
    ["Name", "Sex", "Age", "Pclass", "FamilySize", "Fare", TARGET, "score"]
].iloc[[0, 1, 2, -3, -2, -1]]

**Findings**

- The best model by CV is **SVM**. On the test it makes 22 FN and 9 FP, so it misses survivors more often than it invents them.
- **By group:**
  - *Men are almost always predicted as dead.* 18 of the 19 errors among men are FN: 9 of 30 in 1st class, 2 of 16 in 2nd, 7 of 72 in 3rd. By title, `Mr` accounts for 17 of the 22 FN. The model has essentially learned "adult man → died". It is right for 84% of men, but it finds only 6 of the 24 male survivors, mostly boys (`Master`).
  - *3rd class women* are the hardest group (4 FN and 6 FP out of 28). Their survival in EDA was 50%, close to a coin flip. Most FP are `Miss`, i.e. young women: 7 of the 9 FP.
  - *1st and 2nd class women* are predicted almost perfectly (1 error each).
- **Examples:**
  - Mr. George Harris (62, 2nd class) and Mr. Ivan Jalsevac (29, 3rd class) survived, although every feature says "died" — luck that the features cannot explain.
  - Mrs. Caram and Miss Yrois are women who died; for women the model says "survived" with high confidence.
  - Miss Helen Allison is a 2-year-old girl from 1st class, the most "saved" profile possible, yet she died together with her mother; only her baby brother survived. Such exceptions cannot be predicted from these features.
- `score` is the signed distance to the SVM boundary: negative means "died", and the larger the absolute value, the more confident the model.

**Check for identifying features.** The same Random Forest is trained with `PassengerId` added.

In [ ]:
leaky = make_pipeline(
    preprocessor(False, [*NUMERIC, "PassengerId"]),
    RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1),
)
leaky_features = [*FEATURES, "PassengerId"]
leaky_cv = cross_val_score(leaky, train[leaky_features], y_train, cv=CV, scoring="f1").mean()
leaky.fit(train[leaky_features], y_train)
leaky_importance = pd.Series(
    leaky[-1].feature_importances_, index=leaky[0].get_feature_names_out()
).sort_values(ascending=False)
print(f"CV F1 without PassengerId: {results.loc['Random Forest', 'CV F1']:.3f}")
print(f"CV F1 with PassengerId:    {leaky_cv:.3f}")
print(f"PassengerId importance rank: {leaky_importance.index.get_loc('PassengerId') + 1}")
leaky_importance.head(5).to_frame("importance").T

**A subtler case: ticket groups.** Families traveled on one ticket, and family members usually survived or died together. 37% of the test passengers share a ticket with someone in train. A model that recognizes a family (by its fare, class, port and family size) can copy the outcome of relatives instead of learning a general rule. The check below removes `Fare` from SVM and, more directly, repeats the CV with folds **grouped by ticket**, so that no family is split between the training and validation folds.

In [ ]:
svm_without_fare = make_pipeline(
    preprocessor(True, [c for c in NUMERIC if c != "Fare"]), SVC(random_state=SEED)
)
no_fare_cv = cross_val_score(svm_without_fare, X_train, y_train, cv=CV, scoring="f1").mean()
shared_ticket = test["Ticket"].isin(train["Ticket"]).mean()
print(f"test passengers sharing a ticket with someone in train: {shared_ticket:.0%}")
print(f"SVM CV F1 with Fare:    {results.loc['SVM', 'CV F1']:.3f}")
print(f"SVM CV F1 without Fare: {no_fare_cv:.3f}")

ticket_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
pd.DataFrame(
    {
        "CV F1, random folds": results.loc[["Decision Tree", "Random Forest", "SVM"], "CV F1"],
        "CV F1, folds grouped by ticket": [
            cross_val_score(
                models[name], X_train, y_train, groups=train["Ticket"], cv=ticket_cv, scoring="f1"
            ).mean()
            for name in ["Decision Tree", "Random Forest", "SVM"]
        ],
    }
)

- `PassengerId` becomes the **2nd most important feature** (0.17), although it is just a row number, yet CV F1 does not change (0.732 → 0.732). The deep trees use the ID to split off individual passengers: memorization without any benefit for new data.
- Impurity importance does not protect against this, which is why `PassengerId`, `Name`, `Ticket` and `Cabin` are excluded.

**Findings: ticket groups**

- **The trees do memorize families.** With folds grouped by ticket, CV F1 drops from 0.728 to 0.674 for the Decision Tree and from 0.732 to 0.698 for Random Forest. Part of their score came from relatives in the training folds. Since 37% of the test passengers also have relatives in train, their test scores are somewhat optimistic too.
- **SVM does not.** Its CV is unchanged (0.749 → 0.754), and without `Fare` it is also unchanged (0.752). Its smooth boundary cannot single out individual families, so its quality comes from general rules (sex, class, age). This is one more reason to prefer it.

## Conclusions

| Model | CV F1 | Test accuracy | Test precision | Test recall | Test F1 |
| --- | --- | --- | --- | --- | --- |
| Baseline | 0.000 | 0.615 | 0.000 | 0.000 | 0.000 |
| Decision Tree | 0.728 | 0.804 | 0.750 | 0.739 | 0.745 |
| Random Forest | 0.732 | 0.804 | 0.758 | 0.725 | 0.741 |
| Gradient Boosting | 0.747 | 0.810 | 0.787 | 0.696 | 0.738 |
| **SVM** | **0.749** | **0.827** | **0.839** | 0.681 | **0.752** |
| KNN | 0.718 | 0.804 | 0.804 | 0.652 | 0.720 |

- All five models pass **F1 > 0.70**. The best by CV and on test is **SVM (F1 0.752)**.
- The main factors are sex/title, then class (and fare as its proxy) and age.
- The unrestricted tree memorizes the training set (train F1 0.98) and loses the most when families are kept out of its training folds. On this 179-passenger test the four best models differ by at most 0.014 in F1, which is within noise, so the choice is made by CV.
- The errors are mostly missed survivors among men and mistakes on 3rd class women, i.e. where survival was close to random.
- Identifying columns are excluded. The checks show that deep trees partly memorize passengers (`PassengerId`) and families (ticket-grouped CV), while the chosen SVM keeps its quality without such information.